In [8]:
import pandas as pd
allowed_df = pd.read_csv("10_matrix.csv", header=None, delimiter=";")
allowed = allowed_df.values.astype(bool)  # 1→True, 0→False dönüşümü

d_df = pd.read_csv("cost_matrix.csv", header=None,delimiter=";")
d = d_df.values
d = d.astype(float)

allowed_df
d_df
print(type(d))

<class 'numpy.ndarray'>


In [9]:
print(allowed.shape)
print(d.shape)
n = len(allowed)
rows_without = [i for i in range(n) if not any(allowed[i, j] for j in range(n))]
cols_without = [j for j in range(n) if not any(allowed[i, j] for i in range(n))]

print("Rows with no allowed j:", rows_without)
print("Cols with no allowed i:", cols_without)

print(range(1,len(allowed)))

(234, 234)
(234, 234)
Rows with no allowed j: []
Cols with no allowed i: []
range(1, 234)


In [10]:
import pulp
import numpy as np

In [11]:
prob = pulp.LpProblem("assign", pulp.LpMinimize)

In [12]:
x = {}
for i in range(len(allowed)):
    for j in range(len(allowed)):
        if allowed[i, j]:
            # continuous veya binary tercihine göre: cat='Binary' vs 'Continuous'
            x[i, j] = pulp.LpVariable(f"x_{i}_{j}", lowBound=0, upBound=1, cat='Binary')

In [ ]:
prob += pulp.lpSum(d[i, j] * x[i, j] for (i, j) in x)

In [ ]:
# Satır (row) kısıtları: her i için toplam 1
for i in range(1,len(allowed)):
    prob += pulp.lpSum(x[i, j] for j in range(len(allowed)) if (i, j) in x) == 1

# Sütun (column) kısıtları: her j için toplam 1
for j in range(1,len(allowed)):
    prob += pulp.lpSum(x[i, j] for i in range(len(allowed)) if (i, j) in x) == 1


In [ ]:
import pulp
try:
    solver = pulp.PULP_CBC_CMD(msg=1)  # CBC solver
    res = prob.solve(solver)
    print("solve() returned:", res)
    print("prob.status (numeric):", prob.status)
    print("prob.status (text):", pulp.LpStatus[prob.status])
except Exception as e:
    print("Solver raised exception:", repr(e))

solution = np.zeros((len(allowed), len(allowed)))
for (i, j), var in x.items():
    solution[i, j] = var.value()


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/bp/2cz163b912j919tv5pxzd0l40000gn/T/4f43b4a7fd0641be9321f0609e745b58-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/bp/2cz163b912j919tv5pxzd0l40000gn/T/4f43b4a7fd0641be9321f0609e745b58-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 471 COLUMNS
At line 126961 RHS
At line 127428 BOUNDS
At line 152820 ENDATA
Problem MODEL has 466 rows, 25391 columns and 50316 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 302229 - 0.06 seconds
Cgl0004I processed model has 456 rows, 25381 columns (25381 integer (25381 of which binary)) and 50299 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied

In [ ]:
print(pulp.LpStatus[prob.status])
print(pulp.value(prob.objective))
x_sol = np.zeros((250, 250))
np.set_printoptions(threshold=np.inf)
for (i, j) in x:
    x_sol[i, j] = pulp.value(x[i, j])

#print(x_sol)
edges = []
k = 1
for (i , j), var in x.items():
    if pulp.value(var) == 1:
        edges.append((i,j))

for (i , j), var in x.items():
    if pulp.value(var) == 1:
        print(i,j)
        

Optimal
302229.0
0 1
0 3
0 5
0 6
0 9
0 76
0 79
0 80
0 81
0 82
0 83
0 131
0 132
0 133
0 145
1 2
2 4
3 77
4 78
5 137
6 138
7 10
8 142
9 12
10 13
11 85
12 150
13 0
14 151
15 157
16 88
17 19
18 102
19 21
20 91
21 161
22 24
23 25
24 26
25 27
26 28
27 29
28 30
29 31
30 32
31 33
32 34
33 35
34 36
35 37
36 103
37 185
38 187
39 189
40 43
41 192
42 193
43 46
44 196
45 197
46 199
47 111
48 112
49 113
50 114
51 54
52 115
53 116
54 117
55 211
56 213
57 214
58 120
59 0
60 62
61 218
62 64
63 220
64 125
65 222
66 126
67 0
68 226
69 70
70 128
71 72
72 232
73 0
74 75
75 0
76 134
77 136
78 8
79 141
80 143
81 146
82 14
83 15
84 16
85 18
86 155
87 156
88 158
89 160
90 100
91 164
92 166
93 168
94 170
95 172
96 174
97 176
98 178
99 180
100 190
101 184
102 186
103 188
104 191
105 194
106 195
107 48
108 200
109 202
110 52
111 53
112 207
113 56
114 57
115 58
116 119
117 215
118 216
119 63
120 0
121 0
122 221
123 223
124 68
125 225
126 0
127 228
128 73
129 233
130 0
131 135
132 7
133 139
134 140
135 11
136 144
1

In [ ]:
from collections import defaultdict

# Adjacency list oluştur
graph = defaultdict(list)
for a,b in edges:
    graph[a].append(b)
    #graph[b].append(a)  # iki yönlü bağlantı kabul edelim (tek yönlü istersen bunu kaldır)

visited = set()
chains = []

def dfs(node, chain):
    visited.add(node)
    chain.append(node)
    for nei in graph[node]:
        if nei not in visited:
            dfs(nei, chain)

# Bağımsız zincirleri bul
for node in graph:
    if node not in visited:
        chain = []
        dfs(node, chain)
        chains.append(chain)
        print(chain)

[0, 1, 2, 4, 78, 8, 142, 3, 77, 136, 144, 87, 156, 162, 95, 172, 108, 200, 55, 211, 217, 124, 68, 226, 71, 72, 232, 5, 137, 84, 16, 88, 158, 182, 105, 194, 201, 208, 61, 218, 66, 126, 6, 138, 147, 154, 92, 166, 97, 176, 40, 43, 46, 199, 206, 59, 9, 12, 150, 20, 91, 164, 96, 174, 101, 184, 106, 195, 51, 54, 117, 215, 123, 223, 227, 230, 76, 134, 140, 149, 89, 160, 94, 170, 99, 180, 104, 191, 110, 52, 115, 58, 120, 79, 141, 86, 155, 23, 25, 27, 29, 31, 33, 35, 37, 185, 44, 196, 203, 210, 121, 80, 143, 17, 19, 21, 161, 167, 173, 179, 39, 189, 109, 202, 209, 81, 146, 153, 159, 165, 171, 177, 183, 42, 193, 50, 114, 57, 214, 219, 67, 82, 14, 151, 90, 100, 190, 198, 205, 118, 216, 65, 222, 69, 70, 128, 73, 83, 15, 157, 163, 169, 175, 181, 41, 192, 49, 113, 56, 213, 131, 135, 11, 85, 18, 102, 186, 107, 48, 112, 207, 60, 62, 64, 125, 225, 229, 130, 132, 7, 10, 13, 133, 139, 148, 22, 24, 26, 28, 30, 32, 34, 36, 103, 188, 47, 111, 53, 116, 119, 63, 220, 224, 129, 233, 145, 152, 93, 168, 98, 178, 